# PPO 与 GRPO

本节介绍 RL 中最重要的两个算法：PPO（Proximal Policy Optimization）和 GRPO（Group Relative Policy Optimization）。我们的 Wordle 训练使用的是 GRPO，但它建立在 PPO 的基础之上，因此需要先理解 PPO。

---

## 1. 优势（Advantage）

在讲 PPO 和 GRPO 之前，先理解一个共同概念：**优势**。

优势衡量的是：某条 rollout 相比一个参照水平好多少（或差多少）。这个参照水平通常叫做 baseline。

- 如果 advantage > 0，说明这条 rollout 比 baseline 好，训练时应该提高类似输出的概率。
- 如果 advantage < 0，说明这条 rollout 比 baseline 差，训练时应该降低类似输出的概率。

不同算法的区别在于：baseline 从哪里来。PPO 通常用 Critic 模型估计 baseline；GRPO 则用同一个 prompt 下多条 rollout 的组内平均奖励作为 baseline。

In [ ]:
# 假设某个 prompt 生成了多条 rollout，并得到了对应奖励
rewards = [1.5, 0.6, 1.8, 0.3, 0.9, 1.2, 0.2, 0.7]
baseline = 0.9  # 先把它理解成“平均水平”或“参照水平”

# 优势 = 奖励 - baseline
# rollout 1: advantage = 1.5 - 0.9 = +0.6  -> 正优势，强化
# rollout 2: advantage = 0.6 - 0.9 = -0.3  -> 负优势，抑制
# rollout 3: advantage = 1.8 - 0.9 = +0.9  -> 正优势，强化
# rollout 8: advantage = 0.2 - 0.9 = -0.7  -> 负优势，抑制


这一节先只关注 advantage 的符号：正优势表示"比参照水平好"，负优势表示"比参照水平差"。至于 baseline 具体怎么得到，PPO 和 GRPO 会采用不同方法。

---

## 2. PPO 算法

PPO 是最经典的策略梯度算法之一。其核心思想是：**限制每次更新的幅度**，防止策略突变。

### 2.1 策略梯度

最基础的策略梯度更新公式：

```text
基础策略梯度
loss = -advantage * log(pi(output))

直觉：advantage > 0 时，增大该输出的概率
       advantage < 0 时，减小该输出的概率
```

### 2.2 PPO Clip

基础策略梯度的问题是：如果某次更新的幅度太大，策略可能突然变差且无法恢复。PPO 引入了 **clip 机制**来限制更新幅度：

```text
PPO Clip 机制

ratio = pi_new(output) / pi_old(output)   # 新旧策略的概率比

if ratio > 1+eps:  clip 到 1+eps  (不要增加太多)
if ratio < 1-eps:  clip 到 1-eps  (不要减少太多)

eps (epsilon) 通常 = 0.2，限制每次更新幅度不超过 20%

loss = -min(ratio * advantage, clip(ratio, 1-eps, 1+eps) * advantage)
```

### 2.3 PPO 的 Critic 问题

标准 PPO 需要一个 **Critic 模型**来估计每个状态的价值（value），再用它作为 baseline 来计算优势。这意味着：

- 额外训练一个与策略模型同等大小的 Critic 模型
- 额外的显存和计算开销
- Critic 训练不好会导致优势估计不准

这在资源受限的环境下（如 2 卡）是一个很大的负担。

---

## 3. GRPO 算法

GRPO（Group Relative Policy Optimization）是 PPO 的改进版，**不需要 Critic 模型**。它保留 PPO 限制更新幅度的思想，但把优势计算方式换成了组内相对比较。

### 3.1 组内相对优势

GRPO 对同一个 prompt 采样多条 rollout，用这些 rollout 的组内平均奖励作为 baseline，再计算每条 rollout 的相对优势：

<img src="./images/grpo_advantage.png" alt="GRPO 组内相对优势" width="80%">

图中的 `baseline` 表示同组 rollout 的平均奖励，用 $\mu$ 表示。每条 rollout 的 reward 会和这个组内平均值比较：高于 $\mu$ 的样本得到正优势，训练时会提高类似回答的概率；低于 $\mu$ 的样本得到负优势，训练时会降低类似回答的概率。

GRPO 通常还会用组内标准差 $\sigma$ 做归一化：

$$
A_i = \frac{r_i - \mu}{\sigma}
$$


```text
GRPO 优势计算 (无 Critic)

对同一个 prompt 生成 N=8 条 rollout
baseline = mean(rewards)  # 组内平均奖励
advantage_i = (reward_i - baseline) / std(rewards)

示例:
  rewards = [1.5, 0.6, 1.8, 0.3, 0.9, 1.2, 0.2, 0.7]
  baseline = mean(rewards) = 0.9
  std = 0.53
  advantage = [1.13, -0.57, 1.70, -1.13, 0.0, 0.57, -1.32, -0.38]

高于 baseline 的 rollout 被强化
低于 baseline 的 rollout 被抑制
不需要 Critic 模型，只需要组内比较
```


这里的关键点是：模型不是在判断某条回答是否“绝对正确”，而是在同一个 prompt 的多次尝试之间做相对比较。

### 3.2 GRPO vs PPO 对比

| 特性 | PPO | GRPO |
|------|-----|------|
| 优势计算 | Critic 模型估计 | 组内平均 |
| 额外模型 | 需要 Critic | 不需要 |
| 显存开销 | 高（两个模型） | 低（一个模型） |
| 适合场景 | 有 Critic 的场景 | 资源受限、规则奖励 |
| Clip 机制 | 有 | 有（继承 PPO） |

### 3.3 为什么 Wordle 适合 GRPO

Wordle 的奖励是**规则计算**的（猜中/部分匹配/格式），不需要 Critic 来估计价值。而且 2 卡环境下显存有限，省掉 Critic 模型意味着可以用更大的 batch size 或更长的序列。

---

## 课后练习

1. （判断题）优势（Advantage）为正的 rollout 会被强化，优势为负的会被抑制。

2. （判断题）PPO 的 clip 机制限制每次策略更新的幅度，防止策略突变。

3. （判断题）GRPO 需要 Critic 模型来估计每个状态的价值。

4. （单选题）PPO 需要额外训练 Critic 模型的主要目的是？
    A. 生成 rollout
    B. 估计状态价值，计算优势
    C. 计算奖励
    D. 计算 KL 散度

5. （单选题）GRPO 相比 PPO 的核心优势是什么？
    A. 训练速度更快
    B. 不需要 Critic 模型，用组内平均代替
    C. 奖励更高
    D. 不需要 clip 机制

6. （多选题）以下哪些是 PPO clip 机制的作用？
    A. 限制新旧策略的概率比
    B. 防止策略突变
    C. 加速收敛
    D. 保证每次更新幅度在安全范围内

In [ ]:
!cat ../cann-learning-hub/tutorials/rl_training_pipeline/02_rl_core_concepts/answer/02.03_answer.txt